### Imports

In [1]:
!java -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)


In [2]:
!pip show pyspark

Name: pyspark
Version: 3.5.5
Summary: Apache Spark Python API
Home-page: https://github.com/apache/spark/tree/master/python
Author: Spark Developers
Author-email: dev@spark.apache.org
License: http://www.apache.org/licenses/LICENSE-2.0
Location: /home/kenuey/AI_Track/ds7-amazon/linux_env/lib/python3.10/site-packages
Requires: py4j
Required-by: 


In [3]:
!pip show graphframes

Name: graphframes
Version: 0.6
Summary: GraphFrames: DataFrame-based Graphs
Home-page: https://github.com/graphframes/graphframes
Author: graphframes
Author-email: modeldb@csail.mit.edu
License: MIT
Location: /home/kenuey/AI_Track/ds7-amazon/linux_env/lib/python3.10/site-packages
Requires: nose, numpy
Required-by: 


In [4]:
!pip show graphframes-py

Name: graphframes-py
Version: 0.11.0
Summary: GraphFrames: Graph Processing Framework for Apache Spark
Home-page: 
Author: GraphFrames Contributors
Author-email: graphframes@googlegroups.com
License: Apache 2.0
Location: /home/kenuey/AI_Track/ds7-amazon/linux_env/lib/python3.10/site-packages
Requires: 
Required-by: 


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.types import *

import os


### Spark Session

In [6]:
os.environ["HADOOP_HOME"] = r"C:\hadoop\hadoop-3.3.6"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\hadoop-3.3.6\bin"

print(os.environ.get("HADOOP_HOME"))
# print(os.environ.get("PATH"))

C:\hadoop\hadoop-3.3.6


In [7]:
import logging

logging.basicConfig(level=logging.INFO)

In [37]:
spark = SparkSession.builder \
    .appName("GraphFrameColab") \
    .master("local[*]") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.9.2") \
    .getOrCreate()

print("Active Spark sessions:", spark.sparkContext.uiWebUrl)

Active Spark sessions: http://10.255.255.254:4040


In [35]:
# spark = (
#     SparkSession.builder
#     .appName("AmazonGraph")
#     .master("local[*]")
#     .config(
#         "spark.jars.packages",
#         # "graphframes:graphframes:0.10.0-spark3.5-s_2.12"
#         "io.graphframes:graphframes-spark4_2.13:0.9.2"
#     )
#     .config("spark.driver.memory", "2g")
#     .config("spark.sql.shuffle.partitions", "50")
#     .getOrCreate()
# )

# print("Active Spark sessions:", spark.sparkContext.uiWebUrl)

In [38]:
AMAZONMETA_TXT = 'data/amazon-meta.txt'
RECOMMS_CSV = 'data/recomms.csv'
MEASURES_JSON = 'data/measures.json'

In [39]:
!head -45 "data/amazon-meta.txt"

IOStream.flush timed out
# Full information about Amazon Share the Love products
Total items: 548552

Id:   0
ASIN: 0771044445
  discontinued product

Id:   1
ASIN: 0827229534
  title: Patterns of Preaching: A Sermon Sampler
  group: Book
  salesrank: 396585
  similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X
  categories: 2
   |Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Preaching[12368]
   |Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Sermons[12370]
  reviews: total: 2  downloaded: 2  avg rating: 5
    2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9
    2003-12-14  cutomer: A2VE83MZF98ITY  rating: 5  votes:   6  helpful:   5

Id:   2
ASIN: 0738700797
  title: Candlemas: Feast of Flames
  group: Book
  salesrank: 168596
  similar: 5  0738700827  1567184960  1567182836  0738700525  0738700940
  categories: 2
   |Books[283155]|Subjects[1000]|Religion 

In [40]:
raw = spark.read.text(AMAZONMETA_TXT)

raw.show(20)

+--------------------+
|               value|
+--------------------+
|# Full informatio...|
| Total items: 548552|
|                    |
|             Id:   0|
|    ASIN: 0771044445|
|  discontinued pr...|
|                    |
|             Id:   1|
|    ASIN: 0827229534|
|  title: Patterns...|
|         group: Book|
|   salesrank: 396585|
|  similar: 5  080...|
|       categories: 2|
|   |Books[283155]...|
|   |Books[283155]...|
|  reviews: total:...|
|    2000-7-28  cu...|
|    2003-12-14  c...|
|                    |
+--------------------+
only showing top 20 rows



In [41]:
rdd = spark.sparkContext.textFile(AMAZONMETA_TXT)

In [42]:
rdd.take(30)

['# Full information about Amazon Share the Love products',
 'Total items: 548552',
 '',
 'Id:   0',
 'ASIN: 0771044445',
 '  discontinued product',
 '',
 'Id:   1',
 'ASIN: 0827229534',
 '  title: Patterns of Preaching: A Sermon Sampler',
 '  group: Book',
 '  salesrank: 396585',
 '  similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X',
 '  categories: 2',
 '   |Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Preaching[12368]',
 '   |Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Sermons[12370]',
 '  reviews: total: 2  downloaded: 2  avg rating: 5',
 '    2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9',
 '    2003-12-14  cutomer: A2VE83MZF98ITY  rating: 5  votes:   6  helpful:   5',
 '',
 'Id:   2',
 'ASIN: 0738700797',
 '  title: Candlemas: Feast of Flames',
 '  group: Book',
 '  salesrank: 168596',
 '  similar: 5  0738700827  1567184960  1567182836  0738

In [43]:
rdd = rdd.zipWithIndex().filter(lambda x: x[1] > 1).map(lambda x: x[0])

INFO:py4j.clientserver:Error while sending or receiving.
Traceback (most recent call last):
  File "/home/kenuey/AI_Track/ds7-amazon/linux_env/lib/python3.10/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
INFO:py4j.clientserver:Closing down clientserver connection
INFO:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/kenuey/AI_Track/ds7-amazon/linux_env/lib/python3.10/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/kenuey/AI_Track/ds7-amazon/linux_env/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home

In [44]:
rdd.take(30)

['',
 'Id:   0',
 'ASIN: 0771044445',
 '  discontinued product',
 '',
 'Id:   1',
 'ASIN: 0827229534',
 '  title: Patterns of Preaching: A Sermon Sampler',
 '  group: Book',
 '  salesrank: 396585',
 '  similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X',
 '  categories: 2',
 '   |Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Preaching[12368]',
 '   |Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Sermons[12370]',
 '  reviews: total: 2  downloaded: 2  avg rating: 5',
 '    2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9',
 '    2003-12-14  cutomer: A2VE83MZF98ITY  rating: 5  votes:   6  helpful:   5',
 '',
 'Id:   2',
 'ASIN: 0738700797',
 '  title: Candlemas: Feast of Flames',
 '  group: Book',
 '  salesrank: 168596',
 '  similar: 5  0738700827  1567184960  1567182836  0738700525  0738700940',
 '  categories: 2',
 '   |Books[283155]|Subjects[1000]|Religion

In [45]:
rdd.count()

15010572

In [46]:
rdd.getNumPartitions()

30

In [47]:
rdd.glom().map(len).collect()

[516608,
 514252,
 516218,
 514593,
 516118,
 515430,
 512380,
 517043,
 517669,
 515940,
 520971,
 517222,
 515816,
 515597,
 513376,
 518164,
 530414,
 533057,
 519084,
 520509,
 520588,
 519433,
 520296,
 525464,
 522565,
 523157,
 507377,
 491980,
 460324,
 58927]

### Data Preprocessing

In [48]:
from pyspark.sql.types import *
from pyspark.sql import Row

def split_blocks(iter):
    block = []
    for line in iter:
        line = line.strip()
        if line.startswith("Id:") and block:
            yield block
            block = []
        block.append(line)
    if block:
        yield block

In [49]:
def parse_block(block):
    product = {
        "id": None,
        "asin": None,
        "title": "",
        "group": None,
        "salesrank": None,
        "similar_count": 0,
        "similar_asins": []
    }

    for line in block:
        if line.startswith("Id:"):
            product["id"] = line.split()[1]
        elif line.startswith("ASIN:"):
            product["asin"] = line.split()[1]
        elif line.startswith("title:"):
            product["title"] = line.replace("title:", "").strip()
        elif line.startswith("group:"):
            product["group"] = line.split(":")[1].strip()
        elif line.startswith("salesrank:"):
            try:
                product["salesrank"] = int(line.split(":")[1].strip())
            except:
                product["salesrank"] = None
        elif line.startswith("similar:"):
            parts = line.split()
            product["similar_count"] = int(parts[1])
            product["similar_asins"] = parts[2:]

    return [product]

In [50]:
# rdd → blocks → parsed dicts
blocks_rdd = rdd.mapPartitions(split_blocks)
parsed_rdd = blocks_rdd.flatMap(parse_block)

In [51]:
blocks_rdd.take(3)

[[''],
 ['Id:   0', 'ASIN: 0771044445', 'discontinued product', ''],
 ['Id:   1',
  'ASIN: 0827229534',
  'title: Patterns of Preaching: A Sermon Sampler',
  'group: Book',
  'salesrank: 396585',
  'similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X',
  'categories: 2',
  '|Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Preaching[12368]',
  '|Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Sermons[12370]',
  'reviews: total: 2  downloaded: 2  avg rating: 5',
  '2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9',
  '2003-12-14  cutomer: A2VE83MZF98ITY  rating: 5  votes:   6  helpful:   5',
  '']]

In [52]:
parsed_rdd.take(3)

[{'id': None,
  'asin': None,
  'title': '',
  'group': None,
  'salesrank': None,
  'similar_count': 0,
  'similar_asins': []},
 {'id': '0',
  'asin': '0771044445',
  'title': '',
  'group': None,
  'salesrank': None,
  'similar_count': 0,
  'similar_asins': []},
 {'id': '1',
  'asin': '0827229534',
  'title': 'Patterns of Preaching: A Sermon Sampler',
  'group': 'Book',
  'salesrank': 396585,
  'similar_count': 5,
  'similar_asins': ['0804215715',
   '156101074X',
   '0687023955',
   '0687074231',
   '082721619X']}]

In [53]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("title", StringType(), True),
    StructField("group", StringType(), True),
    StructField("salesrank", IntegerType(), True),
    StructField("similar_count", IntegerType(), True),
    StructField("similar_asins", ArrayType(StringType()), True)
])

In [54]:
products_df = spark.createDataFrame(parsed_rdd, schema)

In [55]:
products_df.show(20, truncate=False)

+----+----------+------------------------------------------------------------------------------------------------------------+-----+---------+-------------+------------------------------------------------------------+
|id  |asin      |title                                                                                                       |group|salesrank|similar_count|similar_asins                                               |
+----+----------+------------------------------------------------------------------------------------------------------------+-----+---------+-------------+------------------------------------------------------------+
|NULL|NULL      |                                                                                                            |NULL |NULL     |0            |[]                                                          |
|0   |0771044445|                                                                                                            |NU

In [56]:
products_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- group: string (nullable = true)
 |-- salesrank: integer (nullable = true)
 |-- similar_count: integer (nullable = true)
 |-- similar_asins: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [57]:
vertices = products_df.select(
    col("asin").alias("id"),
    products_df.asin.alias("id"),
    "title",
    "group",
    "salesrank"
).distinct()#.filter("id is NOT NULL")

In [58]:
# from pyspark.sql.functions import col

# vertices = (
#     products_df
#     .select(
#         col("asin").alias("id"),
#         "title",
#         "group",
#         "salesrank"
#     )
#     .filter(col("id").isNotNull())
# )

In [59]:
vertices.show(5, truncate=False)

+----------+----------+---------------------------------------------------------------------------------------------------------------------------------+-----+---------+
|id        |id        |title                                                                                                                            |group|salesrank|
+----------+----------+---------------------------------------------------------------------------------------------------------------------------------+-----+---------+
|037575380X|037575380X|Black No More : A Novel (Modern Library (Paperback))                                                                             |Book |251469   |
|B00000IC83|B00000IC83|Life of Jesus, Vol. 1-2                                                                                                          |DVD  |38774    |
|0553487388|0553487388|The Icicle Forest (Fairy School)                                                                                               

In [60]:
from pyspark.sql.functions import explode, col

edges = products_df.select(
    col("asin").alias("src"),
    explode(col("similar_asins")).alias("dst")
)

In [61]:
edges.show(5)

+----------+----------+
|       src|       dst|
+----------+----------+
|0827229534|0804215715|
|0827229534|156101074X|
|0827229534|0687023955|
|0827229534|0687074231|
|0827229534|082721619X|
+----------+----------+
only showing top 5 rows



In [63]:
from graphframes import GraphFrame

g = GraphFrame(vertices, edges)

Py4JJavaError: An error occurred while calling o292.createGraph.
: java.lang.NoClassDefFoundError: scala/collection/ArrayOps$
	at org.graphframes.GraphFrame$.apply(GraphFrame.scala:767)
	at org.graphframes.GraphFramePythonAPI.createGraph(GraphFramePythonAPI.scala:9)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnect

In [ ]:
# # Vertices
# vertices = products_df.selectExpr("asin as id", "title")

# # Edges
# from pyspark.sql.functions import explode, col

# edges = products_df \
#     .withColumn("dst", explode(col("similar_asins"))) \
#     .select(col("asin").alias("src"), col("dst"))

# edges.show(5)

In [52]:
# vertices = products_df.selectExpr("asin as id", "title", "group", "salesrank")


# edges = products_df.rdd.flatMap(lambda row: [
#     Row(src=row.asin, dst=asin) for asin in row.similar_asins
# ]).toDF()


In [53]:
# g = GraphFrame(vertices, edges)

# # Пример: показать вершины и ребра
# print("Vertices:")
# g.vertices.show(5, truncate=False)

# print("Edges:")
# g.edges.show(5, truncate=False)


In [54]:
# # Пример: количество соседей для каждого товара
# g.degrees.show(5)

# # PageRank (важность товара в графе)
# results = g.pageRank(resetProbability=0.15, maxIter=5)
# results.vertices.select("id", "pagerank").show(5)


### Descriptive Analysis

### Bundles and Collections

### Graph Visualization

### New Recommender System